# **Universidad de Buenos Aires**
# CEIA - Aprendizaje por Refuerzo I
# Tema: SARSA y Q-Learning en CliffWalking
---
# Trabajo Práctico 2

De acuerdo a la siguiente celda resolver y responder las preguntas modificando el código original y reescribiéndolo luego de cada pregunta.


(En esta celda se implementan los algoritmos **SARSA** y **Q-Learning** originales de Sutton & Barto, aplicados al entorno **CliffWalking** de `gymnasium`, y se comparan sus resultados. A lo largo del TP se irán realizando ajustes progresivos hasta lograr que ambos aprendan una política satisfactoria.)

In [ ]:
# ============================================================
# Celda 1: Instalación e imports
# ============================================================
!pip install gymnasium matplotlib numpy -q

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ============================================================
# Celda 2: Entorno y parámetros
# ============================================================
env = gym.make('CliffWalking-v1')

N_STATES = env.observation_space.n     # 48 estados (grilla 4x12)
N_ACTIONS = env.action_space.n         # 4 acciones: 0 arriba, 1 derecha, 2 abajo, 3 izquierda

NUM_EPISODES = 500
ALPHA = 0.5
GAMMA = 1.0
EPSILON = 0.3   # epsilon CONSTANTE (sin decaimiento) durante todo el entrenamiento

# ============================================================
# Celda 3: Inicialización de las tablas Q
# ============================================================
Q_sarsa = np.zeros((N_STATES, N_ACTIONS))
Q_qlearning = np.zeros((N_STATES, N_ACTIONS))

# ============================================================
# Celda 4: Política de comportamiento epsilon-greedy (común a ambos)
# ============================================================
def epsilon_greedy(Q, state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[state]))

def greedy(Q, state):
    return int(np.argmax(Q[state]))

# ============================================================
# Celda 5: Algoritmo SARSA (on-policy, Sutton & Barto)
# ============================================================
def run_sarsa_episode(Q, alpha, gamma, epsilon):
    state, _ = env.reset()
    action = epsilon_greedy(Q, state, epsilon)
    total_reward = 0
    done = False
    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_action = epsilon_greedy(Q, next_state, epsilon)

        # Actualización SARSA: usa la acción REALMENTE tomada en s' (on-policy)
        Q[state][action] += alpha * (
            reward + gamma * Q[next_state][next_action] - Q[state][action]
        )

        state, action = next_state, next_action
        total_reward += reward
    return total_reward

# ============================================================
# Celda 6: Algoritmo Q-Learning (off-policy, Sutton & Barto)
# ============================================================
def run_qlearning_episode(Q, alpha, gamma, epsilon):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = epsilon_greedy(Q, state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Actualización Q-Learning: usa el máximo sobre s' (off-policy)
        best_next = np.max(Q[next_state])
        Q[state][action] += alpha * (
            reward + gamma * best_next - Q[state][action]
        )

        state = next_state
        total_reward += reward
    return total_reward

# ============================================================
# Celda 7: Entrenamiento de ambos agentes
# ============================================================
rewards_sarsa = []
rewards_qlearning = []
epsilon = EPSILON

for ep in range(NUM_EPISODES):
    r_sarsa = run_sarsa_episode(Q_sarsa, ALPHA, GAMMA, epsilon)
    r_qlearning = run_qlearning_episode(Q_qlearning, ALPHA, GAMMA, epsilon)

    rewards_sarsa.append(r_sarsa)
    rewards_qlearning.append(r_qlearning)

    if (ep + 1) % 50 == 0:
        avg_s = np.mean(rewards_sarsa[-50:])
        avg_q = np.mean(rewards_qlearning[-50:])
        print(f"Ep {ep+1} | ε={epsilon:.3f} | SARSA (últ. 50): {avg_s:.1f} | Q-Learning (últ. 50): {avg_q:.1f}")

# ============================================================
# Celda 8: Visualización comparativa
# ============================================================
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(moving_average(rewards_sarsa), color='blue', label='SARSA')
plt.plot(moving_average(rewards_qlearning), color='orange', label='Q-Learning')
plt.xlabel('Episodio')
plt.ylabel('Recompensa acumulada (media móvil)')
plt.title('SARSA vs Q-Learning en CliffWalking')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ============================================================
# Celda 9: Evaluación y políticas finales
# ============================================================
def evaluate_policy(Q, n_eval=1, max_steps=200):
    """Ejecuta la política greedy (100% explotación) y devuelve la recompensa total."""
    total = 0
    for _ in range(n_eval):
        state, _ = env.reset()
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = greedy(Q, state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total += reward
            steps += 1
    return total / n_eval

print(f"\n✅ Recompensa de la política final SARSA:      {evaluate_policy(Q_sarsa):.1f}")
print(f"✅ Recompensa de la política final Q-Learning: {evaluate_policy(Q_qlearning):.1f}")

ARROWS = {0: '↑', 1: '→', 2: '↓', 3: '←'}
N_ROWS, N_COLS = 4, 12
CLIFF = set(range(37, 47))  # fila inferior, entre el inicio (36) y la meta (47)
START, GOAL = 36, 47

def print_policy(Q, name):
    print(f"\nPolítica aprendida — {name} (↑ 0, → 1, ↓ 2, ← 3):")
    for row in range(N_ROWS):
        line = ""
        for col in range(N_COLS):
            s = row * N_COLS + col
            if s == START:
                line += "🤖 "
            elif s == GOAL:
                line += "🎯 "
            elif s in CLIFF:
                line += "💧 "
            else:
                line += ARROWS[greedy(Q, s)] + " "
        print(line)

print_policy(Q_sarsa, "SARSA")
print_policy(Q_qlearning, "Q-Learning")

**1.** Imprima la Q-table completa de `Q_sarsa` y de `Q_qlearning` al finalizar el entrenamiento (o al menos las filas correspondientes a los estados de la fila inmediatamente superior al acantilado, estados 25 a 34). Observe puntualmente el valor Q(s, "abajo") —la acción que lleva directo al precipicio— en esos estados.

¿Por qué ese valor es mucho más negativo en la tabla de SARSA que en la de Q-Learning para los mismos estados? Relacione la respuesta con cómo cada algoritmo calcula el valor del próximo par estado-acción al actualizar Q (la acción efectivamente tomada por la política de comportamiento vs. la mejor acción posible). (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 1: imprima Q_sarsa y Q_qlearning (o las filas de estados 25 a 34) y
# compare el valor Q(s, 'abajo') entre ambas tablas


Respuesta

**2.** Modifique el código para que `EPSILON = 0` durante todo el entrenamiento (sin exploración: la política de comportamiento es 100% greedy desde el episodio 1).

¿Qué ocurre con las políticas finales de SARSA y de Q-Learning en este caso? ¿Convergen ahora a la misma política? ¿Por qué desaparece la diferencia entre ambos algoritmos cuando no hay exploración? (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 2: modifique acá el código original fijando EPSILON = 0 durante todo
# el entrenamiento y vuelva a entrenar ambos agentes


Respuesta

**3.** Sin modificar el resto del código, agregue el registro de la recompensa acumulada promedio de los últimos 100 episodios **durante el entrenamiento** (no en la evaluación final greedy) para ambos algoritmos, usando `EPSILON = 0.3` constante.

¿Cuál de los dos algoritmos obtiene, en promedio, mejor recompensa mientras todavía se está entrenando (es decir, incluyendo las caídas al acantilado producto de la exploración)? Explique por qué SARSA puede tener mejor desempeño "en línea" durante el entrenamiento aunque termine aprendiendo una política final más larga y conservadora. (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 3: agregue el registro de la recompensa acumulada promedio durante
# el entrenamiento (no en la evaluación greedy) para SARSA y Q-Learning


Respuesta

**4.** Imprima puntualmente los valores Q(36, a) para las 4 acciones posibles desde el estado inicial (36), en ambas tablas.

¿La acción con mayor valor Q en SARSA es la misma que en Q-Learning? ¿Cómo se relaciona esto con la primera decisión que toma cada agente al iniciar un episodio (alejarse de entrada del acantilado o intentar bordearlo)? (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 4: imprima puntualmente Q_sarsa[36] y Q_qlearning[36] y compare cuál
# es la acción con mayor valor Q en cada caso


Respuesta

**5.** Durante los primeros episodios de entrenamiento, con `EPSILON` alto, ambos agentes caen muchas veces al acantilado. Inspeccione cómo cambia con las corridas el valor Q(s, "abajo") de un estado cercano al acantilado en `Q_qlearning` a medida que aumenta la cantidad de episodios (imprímalo, por ejemplo, cada 100 episodios).

¿Ese valor queda "marcado" para siempre por las malas experiencias iniciales, o el algoritmo logra corregirlo a medida que seguimos actualizando Q? Justifique la respuesta observando la ecuación de actualización de Q-Learning. (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 5: imprima cómo evoluciona Q_qlearning[s]['abajo'] para un estado
# cercano al acantilado cada 100 episodios de entrenamiento


Respuesta

**6.** Modifique la función `evaluate_policy` para que, en lugar de actuar 100% greedy, la evaluación final se haga con una política ε-greedy residual (por ejemplo `epsilon=0.05`), igual que hace la política de comportamiento durante el entrenamiento.

¿Qué le pasa a la recompensa obtenida por la política de Q-Learning frente a la de SARSA cuando queda algo de exploración residual en la evaluación? ¿Por qué SARSA resulta más robusto ante este tipo de evaluación que Q-Learning? (VALOR EJERCICIO 0.4)

In [ ]:
# Pregunta 6: modifique evaluate_policy para que use una política epsilon-greedy
# residual (por ejemplo epsilon=0.05) en lugar de 100% greedy


Respuesta

**7.** Ajuste los hiperparámetros que considere necesarios (inicialización de Q, `ALPHA`, `EPSILON` y/o un esquema de decaimiento, `NUM_EPISODES`) hasta lograr que **ambos** algoritmos aprendan una política razonable. Imprima entonces la Q-table completa de ambos y, para cada estado de la fila adyacente al acantilado, identifique cuál es la acción con mayor valor Q en cada tabla.

Con esa evidencia concreta (los valores de la Q-table, no solo el dibujo de la política), explique en sus propias palabras por qué SARSA, al ser **on-policy**, "sabe" que va a seguir explorando ocasionalmente y por eso penaliza más los estados cercanos al precipicio, mientras que Q-Learning, al ser **off-policy**, asume que de ahora en más actuará siempre de forma greedy y por eso no penaliza esos estados de la misma manera. (VALOR EJERCICIO 0.9)

In [ ]:
# Pregunta 7: ajuste los hiperparámetros necesarios, entrene ambos agentes,
# imprima las Q-tables completas y compare la acción de mayor valor en los
# estados junto al acantilado


Respuesta